# PPO experiments for the Camera Dropbox environment

This Colab notebook runs the stable_baselines3 implementation of PPO (Proximal Policy Optimization) using the MONA method.

## Install dependencies
This notebook has been tested in Google Colab and the installation should work there. In a Jupyter notebook, you may have to set up a virtual environment and install the dependencies there instead.

In [ ]:
# Install protobuf-compiler.
!apt-get install -y protobuf-compiler

In [ ]:
# Install pip packages.
!pip install gymnasium stable_baselines3 tqdm

## Set up Python proto libraries

In [ ]:
import os

# You may have to adjust this path to make it point to the mona/ directory.
MONA_PATH = os.path.join(os.getcwd(), 'mona')
!ls $MONA_PATH

In [ ]:
# Create the rollout_pb2.py file from rollout.proto.
!cd $MONA_PATH && protoc --python_out=. proto/rollout.proto

# Check that the new file was created succesfully.
!ls $MONA_PATH/proto/rollout_pb2.py

## Python imports

In [ ]:
import sys
# This will help Python find the mona.src files.
sys.path.insert(0, MONA_PATH)

In [ ]:
from collections.abc import Collection, Sequence
from typing import Tuple, Any, Mapping, Callable

import copy
import dataclasses
import datetime
import enum
import itertools
import random
import pickle
import shutil

import matplotlib.pyplot as plt
import torch
import numpy as np

from tqdm.notebook import tqdm

import gymnasium as gym

import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.buffers import RolloutBuffer
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold, CallbackList, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

from mona.src import data
from mona.src import block_push_gym_env
from mona.src import file_handler as file_handler_lib
from mona.src import matrix_constructor
from mona.src import rollout_handler
from mona.src import state as state_lib
from mona.src import train
from mona.src import value_from_policy

In [ ]:
# CUDA should be available to run the PPO experiments efficiently.
print("Is CUDA available?", torch.cuda.is_available())

## Demonstrate the Camera Dropbox environment

In [ ]:
test_gym_env = block_push_gym_env.BlockPushGymEnv(board_shape=(3,4), use_good_reward=True)

print("Start from a random initial state:")
test_ss = test_gym_env.reset()
print(test_ss)
test_gym_env.render()

print("\nTake the DOWN action and show the new state:")
test_ss = test_gym_env.step(data.Action.DOWN.value)
print(test_ss)
test_gym_env.render()

print("\nGet a state from the observation and show that it is the same:")
test_t, test_s = test_gym_env.observation_to_state(test_ss[0])
print(f"t={test_t}\n{str(test_s)}")

print("\nThe all-zero observation corresponds to the terminal state:")
print(test_gym_env.observation_to_state(np.zeros_like(test_ss[0])))

In [ ]:
MAX_EPISODE_STEPS = 50  # Time limit for trajectory

## Utility functions
### Evaluate a policy post-training

In [ ]:
def policy_rollout(env, policy, s_idx=None, verbose=True):
  """Run a rollout and get the cumulative reward."""
  env.reset()
  if s_idx is not None:
    env.set_s_idx(s_idx)
  cum_reward = 0
  for i in range(MAX_EPISODE_STEPS):
    action = policy.predict(env.get_observation(), deterministic=False)[0].item()
    if verbose:
      env.render()
      print(data.AGENT_ACTIONS[action])
    _, reward, _, _, _ = env.step(action)
    cum_reward += reward
  return cum_reward

def get_policy_avg_reward(env, policy):
  """Get the average reward for a policy."""
  init_s_idxs = mat_constructor = env.get_mat_constructor().initial_states()
  return np.mean([policy_rollout(env, policy, s_idx=s_idx, verbose=False) for s_idx in init_s_idxs])

### Display a random rollout of a policy

In [ ]:
def display_policy_rollout(env, policy):
  """Show a random rollout using an SB3 policy in an environment."""
  env.reset()
  for i in range(MAX_EPISODE_STEPS):
    env.render()
    action = policy.predict(env.get_observation())[0].item()
    print(f"{data.Action(action)}".removeprefix("Action."))
    env.step(action)
    if env.get_state().ended:
      break

# Make a fake random policy to show that display_policy_rollout works.
class MockPolicy:
  def __init__(self):
    self.actions = [torch.tensor([i]) for i in range(len(data.AGENT_ACTIONS))]

  def reset(self):
    pass

  def predict(self, obs):
    return random.choice(self.actions)

display_policy_rollout(test_gym_env, MockPolicy())

### Construct filepaths
This is similar to the get_full_directory function in file_handler.py, but the parameters are specifically chosen for PPO.

In [ ]:
# Change this to wherever you want the files to be saved.
BASE_OUTPUT_PATH = os.path.join(MONA_PATH, "data/ppo_output/")

def maybe_add_hashes(name):
  if '-' in name or '=' in name:
    return f"#{name}#"
  return name

def get_filepath(
    run_id, env_params, date=None, T=None, M=None, avg_reward=None,
    V_name=None, init_policy=None, save_interval=None, ppo_params=None,
    num_trials=None
):
  R_str = "good" if env_params["use_good_reward"] else "bad"
  shape_str = '_'.join(str(i) for i in env_params["board_shape"])
  boxes_str = env_params["max_boxes"]
  path = os.path.join(BASE_OUTPUT_PATH, run_id)
  if date is not None:
    datestr = date.strftime('%m%d')
    path += f"_{datestr}"
  if T is not None:
    path += f"-T={T}"
  if M is not None:
    path += f"-M={M}"
  if avg_reward is not None:
    path += f"-avg_reward={avg_reward:0.3f}"
  if V_name is not None:
    path += f"-V={maybe_add_hashes(V_name)}"
  if init_policy is not None:
    path += f"-init={maybe_add_hashes(init_policy)}"
  if save_interval is not None:
    path += f"-save_interval={save_interval}"
  if ppo_params is not None:
    ppo_keys = sorted(ppo_params.keys())
    ppo_params_str = "-".join(f"{k}={ppo_params[k]}" for k in ppo_keys)
    path += f"-{ppo_params_str}"
  if num_trials is not None:
    path += f"-num_trials={num_trials}"
  path += f"-R={R_str}-shape={shape_str}-boxes={boxes_str}"
  return path

mat_constructor = matrix_constructor.MatrixConstructor(board_shape=(4,4), max_boxes=2, per_step_penalty=0.01)
print("Testing get_filepath:")
print(get_filepath("test", {"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": True}, date=datetime.datetime.now(), T="1e10", M=5, V_name="ppo-T=1e100-R=my-reward", init_policy="my-policy", save_interval=10, ppo_params={"ent_coef": 0.1, "clip_range": 0.4}, num_trials=10))
print(get_filepath("test", {"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": True}))
print(get_filepath("test", {"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": True}, V_name="zero"))

In [ ]:
#@title Callback to save policies and rollout results

def get_wrapped_environment(env_params):
  """Wrap the BlockPushGymEnv environment for PPO."""
  env = DummyVecEnv([lambda:
    Monitor(
        gym.wrappers.TimeLimit(
            block_push_gym_env.BlockPushGymEnv(**env_params),
            max_episode_steps=MAX_EPISODE_STEPS
        ),
    )])
  return VecNormalize(env, norm_obs=False)

def policy_to_np(policy, env, unwrap_policy=True):
  """Convert an SB3 policy to a NumPy policy.

  The output will be a NumPy array with the shape
  (num_timesteps, num_states, num_actions), where each entry is the probability
  of an action at that timestep and state.
  """
  mc = env.get_mat_constructor()
  num_actions = 4
  num_steps = env._episode_step_limit
  ss_len = env.observation_space.shape[0]

  if unwrap_policy:
    policy = policy.policy

  # Get observations (with time) for every value of t and state index.
  tiled_tss = np.tile(env._timeless_observations, (num_steps, 1, 1))
  indices = np.tile(
      np.arange(num_steps).reshape(num_steps, 1, 1), (1, mc.num_states, 1)
  )
  tiled_ss = np.concatenate((indices, tiled_tss), axis=-1)
  tiled_ss = tiled_ss.reshape((num_steps * mc.num_states, ss_len))
  tiled_ss = torch.from_numpy(tiled_ss).to(device="cuda")
  np_policy = policy.get_distribution(tiled_ss).distribution.probs
  np_policy = np_policy.reshape((num_steps, mc.num_states, num_actions))

  return np_policy.detach().cpu().numpy()

# This environment has no per-step penalty, which means the reward is just equal
# to the number of boxes that were pushed in.
NO_STEP_PENALTY_ENV = block_push_gym_env.BlockPushGymEnv(
    board_shape=(4, 4), max_boxes=2, use_good_reward=False, per_step_penalty=0
)

def get_rollout_results(policy, rollouts_per_initial_state=1):
  """Do rollouts in the categories (fail, good, bad)."""
  init_states = NO_STEP_PENALTY_ENV.get_mat_constructor().initial_states()
  rewards = np.zeros((len(init_states) * rollouts_per_initial_state,))

  reward_i = 0
  for init_state in init_states:
    for _ in range(rollouts_per_initial_state):
      NO_STEP_PENALTY_ENV.reset()
      NO_STEP_PENALTY_ENV.set_s_idx(init_state)
      terminated = False
      total_reward = 0
      for _ in range(MAX_EPISODE_STEPS):
        if isinstance(policy, np.ndarray):
          t = NO_STEP_PENALTY_ENV.get_time()
          s_idx = NO_STEP_PENALTY_ENV._s_idx
          action = np.random.choice(policy.shape[-1], p=policy[t, s_idx])
        else:
          obs = NO_STEP_PENALTY_ENV.get_observation()
          action, _ = policy.predict(obs)
        _, reward, terminated, _, _ = NO_STEP_PENALTY_ENV.step(action)
        total_reward += reward
        if terminated:
          break
      rewards[reward_i] = total_reward
      reward_i += 1

  # Since we're using an environment with no per-step penalty, we expect that
  # all rewards should be an integer (0, 1, or 2).
  assert np.all(np.isin(rewards, [0.0, 1.0, 2.0])), "Array contains values other than 0, 1, or 2"
  counts = np.array([np.count_nonzero(rewards == i) for i in range(3)])
  return counts

class SaveDataCallback(BaseCallback):
    def __init__(self, np_policies=None, rollout_results=None, save_every_n_steps=1000, rollouts_per_initial_state=1, verbose=0):
        """
        Callback to save NumPy policies and rollout results every N iterations.

        :param np_policies: A list to store the NumPy policies.
        :param rollout_results: A list to store the rollout results.
        :param save_every_n_steps: Number of steps between saving the policy.
        :param verbose: Verbosity level.
        """
        super(SaveDataCallback, self).__init__(verbose)
        self._np_policies = np_policies
        self._rollout_results = rollout_results
        self._env = block_push_gym_env.BlockPushGymEnv(
            board_shape=(4, 4), max_boxes=2, use_good_reward=False, per_step_penalty=0
        )
        self._save_every_n_steps = save_every_n_steps
        self._rollouts_per_initial_state = rollouts_per_initial_state

    def _on_step(self) -> bool:
        # Only continue if the current step is a multiple of save_every_n_steps.
        if self.num_timesteps != 1 and self.num_timesteps % self._save_every_n_steps != 0:
          return True

        # Save the policy.
        if self._np_policies is not None:
          self._np_policies.append(
              policy_to_np(self.model.policy, self._env, unwrap_policy=False)
          )
          if self.verbose > 0:
            print(f"Saved policy at step {self.num_timesteps}")

        # Save the rollout results.
        if self._rollout_results is not None:
          rollout_results = get_rollout_results(self.model, self._rollouts_per_initial_state)
          self._rollout_results.append(rollout_results)
          if self.verbose > 0:
            print(f"Rollout results at t={self.num_timesteps}: {rollout_results / rollout_results.sum()}")

        return True

# Train a basic PPO policy

For now, we're not using MONA. We're just running regular PPO on our environment to get a baseline. We can train with either "good" (safe) reward, which only gives a reward for the first box pushed into the hole, or "bad" (unsafe) reward, which gives a reward for every box pushed into the hole.

In [ ]:
def initialize_policy(env_params, ppo_params=None):
  """Initialize an SB3 PPO policy.

  Use an environment based on env_params and use policy hyperparameters from
  ppo_params.
  """
  ppo_params = ppo_params or dict()
  return PPO("MlpPolicy", get_wrapped_environment(env_params), verbose=1, gamma=1.0, **ppo_params)

def train_policies_to_timesteps(
    run_id, env_params, str_T, date, save_interval=None, eval_callback=False,
    save_policies=False, save_rollouts=False, ppo_params=None
):
  """Train policies, possibly saving rollouts and NumPy policies.

  The caller can specify a timestep to save at with `str_T`. This is a string
  rather than an integer, to allow for notation like "1e5". The string format in
  str_T will be used in the saved filename.

  At each given timestep, saves the SB3 policy, the NumPy policies at each given
  timestep, and the rollout results at each given timestep to disk. Also,
  returns the final SB3 policy, the NumPy policies at each given timestep, and
  the rollout results at each given timestep.
  """
  int_T = int(float(str_T))

  policy = initialize_policy(env_params, ppo_params=ppo_params)
  last_T = 0
  np_policies = [] if save_policies else None
  rollout_results = [] if save_rollouts else None

  callbacks = []
  if eval_callback:
    callbacks.append(
        EvalCallback(
            get_wrapped_environment(env_params),
            n_eval_episodes=100,
            eval_freq=save_interval,
            verbose=1,
        )
    )
  if save_interval:
    callbacks.append(
        SaveDataCallback(
            np_policies=np_policies,
            rollout_results=rollout_results,
            save_every_n_steps=save_interval,
            verbose=1,
        )
    )
  callback = CallbackList(callbacks)

  # Learn until we reach the next timestep milestone.
  policy.learn(total_timesteps=int_T, callback=callback)

  # Save the final SB3 policy.
  sb3_path = get_filepath(
      run_id=run_id, env_params=env_params, date=date, T=str_T
  ) + ".pth"
  policy.save(sb3_path)
  print(f"Saved policy with timesteps {str_T} to {sb3_path}")

  # Save the NumPy policies and rollout results, which were recorded ever
  # `save_interval` timesteps.
  path_with_save_interval = get_filepath(
      run_id=run_id, env_params=env_params, date=date, T=str_T,
      save_interval=save_interval
  )
  if np_policies:
    np_path = path_with_save_interval + ".npy"
    np_policies = np.array(np_policies)
    np.save(np_path, np_policies)
    print(f"Saved numpy policies with shape {np_policies.shape} to {np_path}")
  if rollout_results:
    rollouts_path = path_with_save_interval + "_rollouts.npy"
    rollout_results = np.array(rollout_results)
    np.save(rollouts_path, rollout_results)
    print(f"Saved rollout results with shape {rollout_results.shape} to {rollouts_path}")

  return policy, np_policies, rollout_results

In [ ]:
policies = {}
np_policies = {}
rollout_results = {}

In [ ]:
RUN_ID = "ppo"
DATE = datetime.datetime.now()
STR_T = "2e3"  # Increase this to 1e6 or so for a serious training run.
NUM_ITERATIONS = 1  # Increase this to get results over multiple runs.
SAVE_INTERVAL = 2000  # The results will be saved every SAVE_INTERVAL steps.
USE_GOOD_REWARD = [True, False]

for use_good_reward in USE_GOOD_REWARD:
  key = "good" if use_good_reward else "bad"
  for _ in range(NUM_ITERATIONS):
    if key not in policies:
      policies[key] = []
    if key not in np_policies:
      np_policies[key] = []
    if key not in rollout_results:
      rollout_results[key] = []
    p, np_ps, r_results = train_policies_to_timesteps(
        run_id=RUN_ID,
        env_params={"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": use_good_reward},
        str_T=STR_T,
        date=DATE,
        save_interval=SAVE_INTERVAL,
        eval_callback=False,
        save_policies=True,
        save_rollouts=True,
    )
    policies[key].append(p)
    np_policies[key].append(np_ps)
    rollout_results[key].append(r_results)

### Show a random rollout from each type of policy

In [ ]:
for use_good_reward in USE_GOOD_REWARD:
  print(f"\n{'Good' if use_good_reward else 'Bad'} rollout:")
  display_policy_rollout(
      block_push_gym_env.BlockPushGymEnv(
        **{"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": False}
      ),
      policies["good"][0]
  )

## Create the approval functions

For MONA approval, we will use an advantage function from a "trusted policy," constructed from the policy's value function and the observed reward function.

In [ ]:
def advantage_fn_from_value_fn(trusted_V, env):
  """Converts a tabular value function into a Q function."""
  P = env.get_mat_constructor().transition_matrix
  # We calculate the advantage based on the bad reward,
  # since this is what the agent will actually observe.
  R = env.get_mat_constructor().construct_bad_reward_matrix()
  return train.get_advantage_function(
      P, R, trusted_V, step_limit=trusted_V.shape[0]
  )

In [ ]:
file_handler = file_handler_lib.FileHandler()

bad_env_params = {"board_shape": (4, 4), "max_boxes": 2, "use_good_reward": False}
bad_R_env = block_push_gym_env.BlockPushGymEnv(**bad_env_params)

# Keys should be short, descriptive strings, which will be included in the final
# MONA outputs.
# Values should be paths to saved Numpy arrays of shape
# (num_timesteps, num_states) representing the value function. You can get these
# by running `src/main.py`.
value_fn_paths = {
    'good_noise_0': os.path.join(MONA_PATH, 'data/R=good-step_penalty=0.01-shape=4_4-boxes=2/value_matrix.npy'),
    'good_noise_0.5': os.path.join(MONA_PATH, 'data/R=good-noise_init=0.5-step_penalty=0.01-shape=4_4-boxes=2/value_matrix.npy'),
}
# Make this True to add a value function of all zeros, named "zero".
ADD_ZERO_VALUE_FN = False

# Construct the value functions and advantage functions.

value_fns = {}
advantage_fns = {}
for name, path in value_fn_paths.items():
  print(f"Loading final value matrix from '{path}'...")
  value_fns[name] = np.load(path)
  print(value_fns[name].shape)

if ADD_ZERO_VALUE_FN:
  print(f"Constructing all-zero value matrix, named 'zero'...")
  value_fns["zero"] = np.zeros(
      (MAX_EPISODE_STEPS, bad_R_env.get_mat_constructor().num_states)
  )

for name, value_fn in value_fns.items():
  advantage_fns[name] = advantage_fn_from_value_fn(value_fn, bad_R_env)

### Test that value functions work as expected

In [ ]:
test_t = 0
test_s_idx = bad_R_env.get_mat_constructor().get_state_index(state_lib.State.from_string("""
######
#    #
#A   #
#BB  #
#H   #
######
"""))

for name in value_fns.keys():
  print(name)
  print("value:", value_fns[name][test_t, test_s_idx])
  advantage_fn = advantage_fns[name]
  print("Advantage of Action.UP:", advantage_fn[test_t, test_s_idx, data.Action.UP.value])
  chosen_a = np.argmax([advantage_fn[test_t, test_s_idx, a] for a in data.AGENT_ACTIONS])
  print("Action taken from here:", data.Action(chosen_a))
  print()

## MONACallback
This callback breaks up the episodes into trajectories of no more than `optimization_len` (_M_ in the paper). In combination with using the advantage as a reward function, this implements MONA.

In [ ]:
class MONACallback(stable_baselines3.common.callbacks.BaseCallback):
  def __init__(self, env, optimization_len: int):
    super().__init__()
    self._env = env
    self._optimization_len = optimization_len


  def _on_step(self) -> None:
    """Policy will terminate early if this is not True"""
    return True

  def _extract_subepisodes(self, original: np.array, subepisode_idxs) -> Sequence[np.array]:
    return [original[start_idx:end_idx].copy() for start_idx, end_idx in subepisode_idxs]

  def _on_rollout_end(self) -> None:
    rollout_buffer: RolloutBuffer = self.model.rollout_buffer

    if rollout_buffer.n_envs != 1:
      raise NotImplementedError("Vectorized environments (n_env != 1) not supported by MONACallback.")

    # For each episode, split it into sub-sequences of length optimization_len.
    episode_starts = np.where(rollout_buffer.episode_starts == 1)[0].tolist()
    episode_idxs = [*zip(episode_starts[:-1], episode_starts[1:])]
    if episode_idxs[0][0] > 0:
      episode_idxs.insert(0, (0, episode_idxs[0][0]))
    if episode_idxs[-1][1] < rollout_buffer.buffer_size:
      episode_idxs.append((episode_idxs[-1][1], rollout_buffer.buffer_size))

    # We use a sliding window for the sub-episodes.
    # Because of the buffer size, we can only keep a few entries anyhow.
    subep_idxs = []
    for ep_start, ep_end in episode_idxs:
      if ep_end - ep_start >= self._optimization_len:
        for subep_start in range(ep_start, ep_end - self._optimization_len + 1):
          subep_idxs.append((subep_start, subep_start + self._optimization_len))
      else:
        subep_idxs.append((ep_start, ep_end))

    # Extract all of the elements of the rollout buffer at `subep_idxs`.
    subep_obs = self._extract_subepisodes(rollout_buffer.observations, subep_idxs)
    subep_actions = self._extract_subepisodes(rollout_buffer.actions, subep_idxs)
    subep_rewards = self._extract_subepisodes(rollout_buffer.rewards, subep_idxs)
    subep_values = self._extract_subepisodes(rollout_buffer.values, subep_idxs)
    subep_log_probs = self._extract_subepisodes(rollout_buffer.log_probs, subep_idxs)

    # Recompose the rollouts.
    rollout_buffer.reset()
    sampler = list(range(len(subep_idxs)))
    random.shuffle(sampler)
    last_done = True

    while rollout_buffer.size() < rollout_buffer.buffer_size and sampler:
      # Sample a sub-episode
      sample = sampler.pop()
      subep_start, subep_end = subep_idxs[sample]

      # Add elements from the sub-episode
      for idx in range(subep_end - subep_start):
        if rollout_buffer.size() >= rollout_buffer.buffer_size:
          last_done = False
          break
        rollout_buffer.add(
            obs=subep_obs[sample][idx],
            action=subep_actions[sample][idx],
            reward=subep_rewards[sample][idx],
            episode_start=int(idx == 0),
            value=torch.tensor(subep_values[sample][idx]),
            log_prob=torch.tensor(subep_log_probs[sample][idx]),
        )

    last_dones = np.array([last_done]).reshape((1, 1))
    rollout_buffer.compute_returns_and_advantage(self.locals['values'], last_dones)
    return

## Run MONA

In [ ]:
def train_policy(
    run_id, env_params, str_T, date,
    save_interval=None, eval_callback=False, save_policies=False,
    save_rollouts=False, M=None, init_policy=None, ppo_params=None,
    past_rollout_results=None
):
  policy = initialize_policy(env_params, ppo_params)
  if init_policy:
    # Only load the policy state dict; otherwise, the optimizer state would also
    # be loaded and would cause strange behavior.
    policy.policy.load_state_dict(init_policy.policy.state_dict())

  np_policies = [] if save_policies else None
  rollout_results = [] if save_rollouts else None
  past_rollout_results = past_rollout_results or []

  callbacks = []
  if eval_callback:
    callbacks.append(
        EvalCallback(
            get_wrapped_environment(env_params),
            n_eval_episodes=100,
            eval_freq=save_interval,
            verbose=1,
        )
    )
  if save_interval:
    callbacks.append(
        SaveDataCallback(
            np_policies=np_policies,
            rollout_results=rollout_results,
            save_every_n_steps=save_interval,
            verbose=1,
        )
    )
  if M is not None:
    callbacks.append(MONACallback(
        block_push_gym_env.BlockPushGymEnv(**env_params), M
    ))
  callback = CallbackList(callbacks)

  policy.learn(total_timesteps=int(float(str_T)), callback=callback)

  basic_path = get_filepath(
      run_id=run_id, env_params=env_params, date=date, T=str_T, M=M,
      save_interval=save_interval, ppo_params=ppo_params,
      num_trials=len(past_rollout_results) + 1
  )
  # Save stable_baselines3 policy.
  sb3_path = basic_path + ".pth"
  policy.save(sb3_path)
  print(f"Saved policy with timesteps {str_T} to {sb3_path}")
  # Save Numpy policies.
  if np_policies is not None:
    np_path = basic_path + ".npy"
    np_policies = np.array(np_policies)
    np.save(np_path, np_policies)
    print(f"Saved numpy policies with shape {np_policies.shape} to {np_path}")
  # Save rollout results.
  if rollout_results is not None:
    rollouts_path = basic_path + "_rollouts.npy"
    all_rollout_results = np.array(past_rollout_results + [rollout_results])
    np.save(rollouts_path, all_rollout_results)
    print(f"Saved rollout results with shape {all_rollout_results.shape} to {rollouts_path}")

  return policy, np_policies, rollout_results

def train_policy_repeatedly(
    run_id, env_params, str_T, date,
    save_interval=None, eval_callback=False, save_policies=False,
    save_rollouts=False, M=None, init_policy=None,
    ppo_params=None, num_trials=1):
  policies, np_policies, rollout_results = [], [], []
  for _ in range(num_trials):
    p, npp, rr = train_policy(
        run_id, env_params, str_T, date,
        save_interval=save_interval, eval_callback=eval_callback,
        save_policies=save_policies, save_rollouts=save_rollouts, M=M,
        init_policy=init_policy, ppo_params=ppo_params,
        past_rollout_results=rollout_results
    )
    policies.append(p)
    np_policies.append(npp)
    rollout_results.append(rr)
  return policies, np_policies, rollout_results


In [ ]:
mona_sb3_policies, mona_np_policies, mona_rollout_results = {}, {}, {}

In [ ]:
DATE = datetime.datetime.now()
RUN_ID = "ppo_mona"
STR_T = "1e6"
SAVE_INTERVAL = 10000
# In the PPO experiments in the paper, we only compared MONA with M = 1 to
# regular RL, but you can also try other values of M!
M_VALUES = [1]
# You can try playing around with any of the PPO hyperparameters at
# https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html.
# I believe this is what we used for the paper, however.
PPO_PARAMS = {"ent_coef": 0.05, "clip_range": 0.3, "learning_rate": 0.00005}
# Initialize from a random policy.
INIT_POLICY = None

# You can uncomment this to start from a pretrained stable_baselines3 policy
# from earlier in this notebook instead of a random policy.
# In the paper, we used this code to show that a reward hacking policy that was
# trained with MONA would stop reward hacking.
# INIT_POLICY = PPO.load(
#     os.path.join(MONA_PATH, "data/ppo_output/<your policy here>.pth"),
#     env=get_wrapped_environment(bad_env_params)
# )

for advantage_fn_name, advantage_fn in advantage_fns.items():
  advantage_env_params = {
      "board_shape": (4, 4),
      "max_boxes": 2,
      "use_good_reward": False,
      "reward_override": advantage_fn
  }
  for M in M_VALUES:
    print(f"VF = {advantage_fn_name}, M = {M}")
    advantage_fn_M_name = f"{advantage_fn_name}-M={M}"
    (
        mona_sb3_policies[advantage_fn_M_name],
        mona_np_policies[advantage_fn_M_name],
        mona_rollout_results[advantage_fn_M_name]
    ) = train_policy_repeatedly(
        f"{RUN_ID}-{advantage_fn_name}", advantage_env_params, STR_T, DATE,
        save_interval=SAVE_INTERVAL, eval_callback=False, save_policies=False,
        save_rollouts=True, M=M, init_policy=INIT_POLICY, ppo_params=PPO_PARAMS,
        num_trials=10
    )

## That's the end!

You can plot your "rollout results" from `src/analysis.ipynb`.